In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. notebooks 폴더 기준 상위(../)의 data/processed 경로로 수정
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
X = df.drop(columns=[target_col])
y = df[target_col]

# 2. 전처리 + 베이스라인(Logistic Regression) 파이프라인
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[("scaler", StandardScaler(), numeric_cols)],
    remainder="passthrough",
)

baseline_model = Pipeline([
    ("prep", preprocessor),
    (
        "clf",
        LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42
        ),
    ),
])

# 3. 5-Fold Stratified K-Fold 교차 검증 (Train vs Val 점검)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_aucs, val_aucs = [], []
f1s, recalls, precisions = [], [], []

for train_idx, val_idx in skf.split(X, y):
  X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
  y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

  baseline_model.fit(X_train, y_train)

  train_prob = baseline_model.predict_proba(X_train)[:, 1]
  val_prob = baseline_model.predict_proba(X_val)[:, 1]
  val_pred = baseline_model.predict(X_val)

  train_aucs.append(roc_auc_score(y_train, train_prob))
  val_aucs.append(roc_auc_score(y_val, val_prob))

  f1s.append(f1_score(y_val, val_pred, zero_division=0))
  recalls.append(recall_score(y_val, val_pred, zero_division=0))
  precisions.append(precision_score(y_val, val_pred, zero_division=0))

print("=== 베이스라인 (Logistic Regression) 5-Fold 결과 ===")
print(f"Train ROC-AUC : {np.mean(train_aucs):.4f}")
print(f"Val ROC-AUC   : {np.mean(val_aucs):.4f} ± {np.std(val_aucs):.4f}")
print(f"AUC Gap       : {(np.mean(train_aucs) - np.mean(val_aucs)):.4f}")
print(f"F1-score      : {np.mean(f1s):.4f}")
print(f"Recall        : {np.mean(recalls):.4f}")
print(f"Precision     : {np.mean(precisions):.4f}")

=== 베이스라인 (Logistic Regression) 5-Fold 결과 ===
Train ROC-AUC : 0.8242
Val ROC-AUC   : 0.8241 ± 0.0028
AUC Gap       : 0.0001
F1-score      : 0.4433
Recall        : 0.7669
Precision     : 0.3118


## 📌 베이스라인(Logistic Regression) 5-Fold 결과 핵심 해석

* **과적합 거의 없음:** `Train ROC-AUC` (`0.8242`)와 `Val ROC-AUC` (`0.8241 ± 0.0028`) 차이가 `0.0001`로 작고 폴드 간 편차도 크지 않아, 훈련 데이터에만 과도하게 맞물리지 않고 일반화 성능이 안정적으로 나타남.

* **판별력 및 기준선 확보:** `Val ROC-AUC` `0.8241`은 무작위 추측(`0.50`)보다 확연히 높으며[cite: 4], 앞으로 파생 변수를 추가하거나 다른 모델을 적용했을 때 *"기존 `0.8241` 기준선을 얼마나 넘어서는가"*를 비교하는 기준점이 됨.

* **불균형 지표 균형(`Recall 0.7669` vs `Precision 0.3118`):** `class_weight="balanced"`를 적용해 실제 당뇨 환자를 포착하는 재현율(`Recall`)이 약 `76.7%`로 높게 나온 대신, 정상인을 당뇨로 예측하는 경우도 있어 정밀도(`Precision`)는 약 `31.2%`로 낮고 `F1-score`는 `0.4433`으로 나타남.

In [3]:
import json
import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. 정제 데이터 로드 (notebooks/ 폴더 기준 상위 ../data/processed 경로)
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col]
X_raw = df.drop(columns=[target_col])


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  """
    4대 축(대사증후군 복합 위험, 연령×대사·건강 취약, 신체기능·심혈관 부담, 불건강 생활습관 누적)
    기반 상호작용 피처를 생성합니다.
    """
  d = data.copy()

  # 보조 플래그
  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  # [1] 대사증후군 복합 위험 축
  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  # [2] 연령 × 대사·건강 취약 결합 축
  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  # [3] 신체기능·심혈관 누적 부담 축
  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  # [4] 불건강 생활습관 누적 및 대사-생활습관 결합 축
  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X_fe = add_interaction_features(X_raw)
added_cols = [c for c in X_fe.columns if c not in X_raw.columns]
print("생성된 파생 변수:", added_cols)


def run_logreg_cv(X_data: pd.DataFrame, y_data: pd.Series):
  num_cols = X_data.select_dtypes(include=[np.number]).columns.tolist()
  preprocessor = ColumnTransformer(
      transformers=[("scaler", StandardScaler(), num_cols)],
      remainder="passthrough",
  )
  pipe = Pipeline([
      ("prep", preprocessor),
      (
          "clf",
          LogisticRegression(
              max_iter=1000, class_weight="balanced", random_state=42
          ),
      ),
  ])

  skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
  tr_aucs, va_aucs, f1s, recs, precs = [], [], [], [], []

  for tr_idx, va_idx in skf.split(X_data, y_data):
    X_tr, X_va = X_data.iloc[tr_idx], X_data.iloc[va_idx]
    y_tr, y_va = y_data.iloc[tr_idx], y_data.iloc[va_idx]

    pipe.fit(X_tr, y_tr)
    tr_prob = pipe.predict_proba(X_tr)[:, 1]
    va_prob = pipe.predict_proba(X_va)[:, 1]
    va_pred = pipe.predict(X_va)

    tr_aucs.append(roc_auc_score(y_tr, tr_prob))
    va_aucs.append(roc_auc_score(y_va, va_prob))
    f1s.append(f1_score(y_va, va_pred, zero_division=0))
    recs.append(recall_score(y_va, va_pred, zero_division=0))
    precs.append(precision_score(y_va, va_pred, zero_division=0))

  return {
      "Train_AUC": np.mean(tr_aucs),
      "Val_AUC": np.mean(va_aucs),
      "Val_AUC_std": np.std(va_aucs),
      "AUC_Gap": np.mean(tr_aucs) - np.mean(va_aucs),
      "F1": np.mean(f1s),
      "Recall": np.mean(recs),
      "Precision": np.mean(precs),
  }


res_raw = run_logreg_cv(X_raw, y)
res_fe = run_logreg_cv(X_fe, y)

compare_df = pd.DataFrame(
    [
        {
            "Feature_Set": "Original_Features",
            "Train_AUC": f"{res_raw['Train_AUC']:.4f}",
            "Val_AUC (mean±std)": (
                f"{res_raw['Val_AUC']:.4f} ± {res_raw['Val_AUC_std']:.4f}"
            ),
            "AUC_Gap": f"{res_raw['AUC_Gap']:.4f}",
            "F1": f"{res_raw['F1']:.4f}",
            "Recall": f"{res_raw['Recall']:.4f}",
            "Precision": f"{res_raw['Precision']:.4f}",
        },
        {
            "Feature_Set": "With_Interaction_Features",
            "Train_AUC": f"{res_fe['Train_AUC']:.4f}",
            "Val_AUC (mean±std)": (
                f"{res_fe['Val_AUC']:.4f} ± {res_fe['Val_AUC_std']:.4f}"
            ),
            "AUC_Gap": f"{res_fe['AUC_Gap']:.4f}",
            "F1": f"{res_fe['F1']:.4f}",
            "Recall": f"{res_fe['Recall']:.4f}",
            "Precision": f"{res_fe['Precision']:.4f}",
        },
    ]
)

print("\n=== [로지스틱 회귀 5-Fold] 원본 vs 파생 변수 추가 비교 ===")
print(compare_df.to_string(index=False))

생성된 파생 변수: ['Metabolic_Risk_Count', 'Triple_Metabolic_Risk', 'Age_HighBP_Inter', 'Aged_Poor_Health', 'Cardio_DiffWalk_Inter', 'GenHlth_BMI_Inter', 'Unhealthy_Lifestyle_Score', 'Metabolic_BadLife_Inter']

=== [로지스틱 회귀 5-Fold] 원본 vs 파생 변수 추가 비교 ===
              Feature_Set Train_AUC Val_AUC (mean±std) AUC_Gap     F1 Recall Precision
        Original_Features    0.8242    0.8241 ± 0.0028  0.0001 0.4433 0.7669    0.3118
With_Interaction_Features    0.8243    0.8241 ± 0.0029  0.0001 0.4425 0.7732    0.3099


## 📌 로지스틱 회귀(5-Fold) 파생 변수 추가 비교 핵심 해석

* **전반적 판별력(`Val ROC-AUC`) 유지:** 파생 변수 8개를 추가해도 `Val ROC-AUC`는 `0.8241 ± 0.0028`에서 `0.8241 ± 0.0029`로 동일하게 유지되었으며, `AUC_Gap` 역시 `0.0001`로 과적합 없이 안정적인 일반화 성능을 보임.

* **재현율(`Recall`) 소폭 상승 vs. 정밀도(`Precision`)·`F1` 미세 하락:** 당뇨 환자를 탐지하는 `Recall`은 `0.7669` → `0.7732` (`+0.63%p`)로 소폭 상승한 반면, `Precision`은 `0.3118` → `0.3099`, `F1-score`는 `0.4433` → `0.4425`로 미세하게 감소함.

* **모델링 인사이트 (선형 모델의 한계와 다음 방향):** 모든 파생 피처가 선형 모델의 성능 점수를 무조건 끌어올리지는 않으며[cite: 3], 기존 원본 변수와 정보가 겹치거나 직선 경계(선형 결합)만으로는 비선형 상호작용 효과를 충분히 살리지 못했기 때문임.

* **다음 실험 전략:** 비선형 분기와 변수 간 결합 패턴을 포착하는 트리 기반 앙상블 모델(`Random Forest`, `XGBoost`, `LightGBM`)에서도 파생 변수가 실질적인 성능 향상과 상위 `Feature Importance`로 이어지는지 비교 검증할 필요가 있음.

In [4]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. 정제된 데이터 로드 (notebooks/ 기준 상위 폴더 경로)
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
X = df.drop(columns=[target_col])
y = df[target_col]

# 2. 전처리 파이프라인 (연속형/수치형 스케일링은 로지스틱 회귀 안정화용)
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[("scaler", StandardScaler(), numeric_cols)],
    remainder="passthrough",
)


def build_models(y_fold_train: pd.Series):
  pos_count = (y_fold_train == 1).sum()
  neg_count = (y_fold_train == 0).sum()
  pos_weight = (neg_count / pos_count) if pos_count > 0 else 1.0

  models = {
      "LogisticRegression (Baseline)": Pipeline([
          ("prep", preprocessor),
          (
              "clf",
              LogisticRegression(
                  max_iter=1000, class_weight="balanced", random_state=42
              ),
          ),
      ]),
      "RandomForest": RandomForestClassifier(
          n_estimators=200,
          max_depth=12,
          min_samples_leaf=20,
          class_weight="balanced",
          n_jobs=-1,
          random_state=42,
      ),
  }

  try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )
  except ImportError:
    print("[안내] xgboost가 설치되지 않아 건너뜁니다.")

  try:
    from lightgbm import LGBMClassifier

    models["LightGBM"] = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
  except ImportError:
    print("[안내] lightgbm이 설치되지 않아 건너뜁니다.")

  return models


# 3. 동일한 5-Fold Stratified K-Fold 분할로 모델 비교
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_names = list(build_models(y).keys())
results = []

for model_name in model_names:
  train_aucs, val_aucs = [], []
  f1s, recalls, precisions = [], [], []

  for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    fold_models = build_models(y_train)
    model = fold_models[model_name]
    model.fit(X_train, y_train)

    train_prob = model.predict_proba(X_train)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]
    val_pred = model.predict(X_val)

    train_aucs.append(roc_auc_score(y_train, train_prob))
    val_aucs.append(roc_auc_score(y_val, val_prob))
    f1s.append(f1_score(y_val, val_pred, zero_division=0))
    recalls.append(recall_score(y_val, y_pred, zero_division=0))
    precisions.append(precision_score(y_val, y_pred, zero_division=0))

  mean_train = np.mean(train_aucs)
  mean_val = np.mean(val_aucs)
  std_val = np.std(val_aucs)

  results.append({
      "Model": model_name,
      "Train_AUC": f"{mean_train:.4f}",
      "Val_AUC (mean±std)": f"{mean_val:.4f} ± {std_val:.4f}",
      "AUC_Gap": f"{(mean_train - mean_val):.4f}",
      "F1-score": f"{np.mean(f1s):.4f}",
      "Recall": f"{np.mean(recalls):.4f}",
      "Precision": f"{np.mean(precisions):.4f}",
  })

summary_df = pd.DataFrame(results)
print("\n=== [3단계] 베이스라인 vs 고급 분류 모델 5-Fold 비교 ===")
print(summary_df.to_string(index=False))

[안내] xgboost가 설치되지 않아 건너뜁니다.
[안내] lightgbm이 설치되지 않아 건너뜁니다.
[안내] xgboost가 설치되지 않아 건너뜁니다.
[안내] lightgbm이 설치되지 않아 건너뜁니다.


NameError: name 'y_pred' is not defined

In [5]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. 정제된 데이터 로드
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
X = df.drop(columns=[target_col])
y = df[target_col]

# 2. 전처리 파이프라인
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[("scaler", StandardScaler(), numeric_cols)],
    remainder="passthrough",
)


def build_models(y_fold_train: pd.Series):
  pos_count = (y_fold_train == 1).sum()
  neg_count = (y_fold_train == 0).sum()
  pos_weight = (neg_count / pos_count) if pos_count > 0 else 1.0

  models = {
      "LogisticRegression (Baseline)": Pipeline([
          ("prep", preprocessor),
          (
              "clf",
              LogisticRegression(
                  max_iter=1000, class_weight="balanced", random_state=42
              ),
          ),
      ]),
      "RandomForest": RandomForestClassifier(
          n_estimators=200,
          max_depth=12,
          min_samples_leaf=20,
          class_weight="balanced",
          n_jobs=-1,
          random_state=42,
      ),
  }

  try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )
  except ImportError:
    pass

  try:
    from lightgbm import LGBMClassifier

    models["LightGBM"] = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
  except ImportError:
    pass

  return models


# 3. 5-Fold Stratified K-Fold 교차 검증
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_names = list(build_models(y).keys())
results = []

for model_name in model_names:
  train_aucs, val_aucs = [], []
  f1s, recalls, precisions = [], [], []

  for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = build_models(y_train)[model_name]
    model.fit(X_train, y_train)

    train_prob = model.predict_proba(X_train)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)  # y_pred 명확히 할당

    train_aucs.append(roc_auc_score(y_train, train_prob))
    val_aucs.append(roc_auc_score(y_val, val_prob))
    f1s.append(f1_score(y_val, y_pred, zero_division=0))
    recalls.append(recall_score(y_val, y_pred, zero_division=0))
    precisions.append(precision_score(y_val, y_pred, zero_division=0))

  mean_train = np.mean(train_aucs)
  mean_val = np.mean(val_aucs)
  std_val = np.std(val_aucs)

  results.append({
      "Model": model_name,
      "Train_AUC": f"{mean_train:.4f}",
      "Val_AUC (mean±std)": f"{mean_val:.4f} ± {std_val:.4f}",
      "AUC_Gap": f"{(mean_train - mean_val):.4f}",
      "F1-score": f"{np.mean(f1s):.4f}",
      "Recall": f"{np.mean(recalls):.4f}",
      "Precision": f"{np.mean(precisions):.4f}",
  })

summary_df = pd.DataFrame(results)
print("\n=== [3단계] 베이스라인 vs RandomForest 5-Fold 비교 ===")
print(summary_df.to_string(index=False))


=== [3단계] 베이스라인 vs RandomForest 5-Fold 비교 ===
                        Model Train_AUC Val_AUC (mean±std) AUC_Gap F1-score Recall Precision
LogisticRegression (Baseline)    0.8242    0.8241 ± 0.0028  0.0001   0.4433 0.7669    0.3118
                 RandomForest    0.8463    0.8277 ± 0.0031  0.0186   0.4426 0.7833    0.3085


## 📌 베이스라인 vs. Random Forest(5-Fold) 비교 핵심 해석

* **일반화 판별력(`Val ROC-AUC`) 개선:** `RandomForest`의 `Val ROC-AUC`가 `0.8277 ± 0.0031`로, 베이스라인(`LogisticRegression`, `0.8241 ± 0.0028`) 대비 `+0.0036` 상승하며 비선형 분기 구조의 장점을 보여줌.

* **과적합(`AUC_Gap`) 관리 상태 양호:** `RandomForest`는 `Train_AUC` `0.8463`, `Val_AUC` `0.8277`로 `AUC_Gap`이 `0.0186`이며, 베이스라인(`0.0001`)보다 차이는 크지만 깊이 제한(`max_depth=12`, `min_samples_leaf=20`)을 두어 과도한 과적합은 억제됨.

* **재현율(`Recall`) 상승 및 `F1`·`Precision` 유지:** 실제 당뇨 환자를 잡아내는 `Recall`이 `0.7669` → `0.7833` (`+0.0164`)으로 높아진 반면, `Precision`은 `0.3118` → `0.3085`, `F1-score`는 `0.4433` → `0.4426`으로 기준선과 유사한 수준을 유지함.

* **모델링 인사이트:** 동일한 조건에서 비교할 때 `RandomForest`가 전체 판별력(`Val ROC-AUC`)과 양성 포착률(`Recall`) 측면에서 기준 모델보다 유리하므로, 파생 변수 효과 확인 및 4단계 `Feature Importance` 대조 분석의 주요 비교 모델로 활용할 수 있음.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

# 1. 정제된 데이터 로드 (notebooks/ 기준 상위 폴더 경로)
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col]
X_raw = df.drop(columns=[target_col])


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  d = data.copy()

  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  # [1] 대사증후군 복합 위험 축
  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  # [2] 연령 × 대사·건강 취약 결합 축
  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  # [3] 신체기능·심혈관 누적 부담 축
  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  # [4] 불건강 생활습관 누적 및 대사-생활습관 결합 축
  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X_fe = add_interaction_features(X_raw)
print(
    "추가된 파생 변수:", [c for c in X_fe.columns if c not in X_raw.columns]
)


def eval_rf_5fold(X_data: pd.DataFrame, y_data: pd.Series):
  skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
  tr_aucs, va_aucs, f1s, recs, precs = [], [], [], [], []

  for tr_idx, va_idx in skf.split(X_data, y_data):
    X_tr, X_va = X_data.iloc[tr_idx], X_data.iloc[va_idx]
    y_tr, y_va = y_data.iloc[tr_idx], y_data.iloc[va_idx]

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )
    rf.fit(X_tr, y_tr)

    tr_prob = rf.predict_proba(X_tr)[:, 1]
    va_prob = rf.predict_proba(X_va)[:, 1]
    va_pred = rf.predict(X_va)

    tr_aucs.append(roc_auc_score(y_tr, tr_prob))
    va_aucs.append(roc_auc_score(y_va, va_prob))
    f1s.append(f1_score(y_va, va_pred, zero_division=0))
    recs.append(recall_score(y_va, va_pred, zero_division=0))
    precs.append(precision_score(y_va, va_pred, zero_division=0))

  return {
      "Train_AUC": np.mean(tr_aucs),
      "Val_AUC": np.mean(va_aucs),
      "Val_AUC_std": np.std(va_aucs),
      "AUC_Gap": np.mean(tr_aucs) - np.mean(va_aucs),
      "F1": np.mean(f1s),
      "Recall": np.mean(recs),
      "Precision": np.mean(precs),
  }


res_raw = eval_rf_5fold(X_raw, y)
res_fe = eval_rf_5fold(X_fe, y)

rf_comp_df = pd.DataFrame(
    [
        {
            "Feature_Set": "RF_Original",
            "Train_AUC": f"{res_raw['Train_AUC']:.4f}",
            "Val_AUC (mean±std)": (
                f"{res_raw['Val_AUC']:.4f} ± {res_raw['Val_AUC_std']:.4f}"
            ),
            "AUC_Gap": f"{res_raw['AUC_Gap']:.4f}",
            "F1": f"{res_raw['F1']:.4f}",
            "Recall": f"{res_raw['Recall']:.4f}",
            "Precision": f"{res_raw['Precision']:.4f}",
        },
        {
            "Feature_Set": "RF_With_Derived",
            "Train_AUC": f"{res_fe['Train_AUC']:.4f}",
            "Val_AUC (mean±std)": (
                f"{res_fe['Val_AUC']:.4f} ± {res_fe['Val_AUC_std']:.4f}"
            ),
            "AUC_Gap": f"{res_fe['AUC_Gap']:.4f}",
            "F1": f"{res_fe['F1']:.4f}",
            "Recall": f"{res_fe['Recall']:.4f}",
            "Precision": f"{res_fe['Precision']:.4f}",
        },
    ]
)

print("\n=== [RandomForest 5-Fold] 원본 vs 파생 변수 8개 추가 비교 ===")
print(rf_comp_df.to_string(index=False))

추가된 파생 변수: ['Metabolic_Risk_Count', 'Triple_Metabolic_Risk', 'Age_HighBP_Inter', 'Aged_Poor_Health', 'Cardio_DiffWalk_Inter', 'GenHlth_BMI_Inter', 'Unhealthy_Lifestyle_Score', 'Metabolic_BadLife_Inter']

=== [RandomForest 5-Fold] 원본 vs 파생 변수 8개 추가 비교 ===
    Feature_Set Train_AUC Val_AUC (mean±std) AUC_Gap     F1 Recall Precision
    RF_Original    0.8463    0.8277 ± 0.0031  0.0186 0.4426 0.7833    0.3085
RF_With_Derived    0.8472    0.8277 ± 0.0029  0.0196 0.4437 0.7783    0.3103


## 분석A RandomForest 원본 vs. 파생 변수 8개 비교 요약

판별력(Val ROC-AUC) 유지: 0.8277 ± 0.0031 → 0.8277 ± 0.0029로 평균은 같고 폴드 편차가 소폭 줄었으며, AUC_Gap은 0.0186 → 0.0196으로 과적합 수준도 비슷함.

지표 변화: F1-score가 0.4426 → 0.4437 (+0.0011), Precision이 0.3085 → 0.3103 (+0.0018)으로 약간 오른 반면, Recall은 0.7833 → 0.7783 (-0.0050)으로 살짝 낮아짐.

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

# 1. 정제된 데이터 로드
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col]
X_raw = df.drop(columns=[target_col])


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  d = data.copy()

  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X_fe = add_interaction_features(X_raw)
derived_cols = [c for c in X_fe.columns if c not in X_raw.columns]

# 2. 5-Fold 평균 Feature Importance 계산
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_importances = []

for tr_idx, _ in skf.split(X_fe, y):
  X_tr, y_tr = X_fe.iloc[tr_idx], y.iloc[tr_idx]
  rf = RandomForestClassifier(
      n_estimators=200,
      max_depth=12,
      min_samples_leaf=20,
      class_weight="balanced",
      n_jobs=-1,
      random_state=42,
  )
  rf.fit(X_tr, y_tr)
  fold_importances.append(rf.feature_importances_)

imp_df = pd.DataFrame({
    "Feature": X_fe.columns,
    "Importance": np.mean(fold_importances, axis=0),
    "Type": [
        "Derived" if c in derived_cols else "Original" for c in X_fe.columns
    ],
}).sort_values("Importance", ascending=False)

print("=== [4단계 연결] RandomForest Feature Importance Top 15 ===")
print(imp_df.head(15).to_string(index=False))

print("\n=== 파생 변수 8개 Importance 순위 ===")
print(imp_df[imp_df["Type"] == "Derived"].to_string(index=False))

=== [4단계 연결] RandomForest Feature Importance Top 15 ===
              Feature  Importance     Type
    GenHlth_BMI_Inter    0.200358  Derived
 Metabolic_Risk_Count    0.154329  Derived
     Age_HighBP_Inter    0.109659  Derived
              GenHlth    0.092381 Original
               HighBP    0.071121 Original
     Aged_Poor_Health    0.065661  Derived
                  Age    0.059530 Original
                  BMI    0.046793 Original
Triple_Metabolic_Risk    0.043883  Derived
             HighChol    0.029719 Original
             DiffWalk    0.022408 Original
               Income    0.018320 Original
 HeartDiseaseorAttack    0.014653 Original
             PhysHlth    0.013530 Original
             MentHlth    0.007318 Original

=== 파생 변수 8개 Importance 순위 ===
                  Feature  Importance    Type
        GenHlth_BMI_Inter    0.200358 Derived
     Metabolic_Risk_Count    0.154329 Derived
         Age_HighBP_Inter    0.109659 Derived
         Aged_Poor_Health    0.065661 De

## 📌 Random Forest 파생 변수 비교 및 Feature Importance(4단계) 핵심 해설

* **파생 변수 8개 추가 시 `RandomForest` 성능 변화:** `Val ROC-AUC`는 `0.8277 ± 0.0031`에서 `0.8277 ± 0.0029`로 동일하게 유지되었으며, `F1-score`(`0.4426` → `0.4437`)와 `Precision`(`0.3085` → `0.3103`)은 소폭 상승하고 `Recall`(`0.7833` → `0.7783`)은 미세하게 감소함.

* **핵심 위험·상호작용 파생 변수의 상위권 장악:** `RandomForest` 5-Fold 평균 `Feature Importance` 상위 1~3위를 `GenHlth_BMI_Inter`(`0.2004`), `Metabolic_Risk_Count`(`0.1543`), `Age_HighBP_Inter`(`0.1097`)가 차지했고, `Aged_Poor_Health`(`0.0657`, 6위)와 `Triple_Metabolic_Risk`(`0.0439`, 9위)까지 총 5개 파생 변수가 Top 15에 진입함.

* **통계·탐색 단계 위험 요인과의 교차 일치:** `GenHlth`(`0.0924`, 4위), `HighBP`(`0.0711`, 5위), `Age`(`0.0595`, 7위), `BMI`(`0.0468`, 8위), `HighChol`(`0.0297`, 10위), `DiffWalk`(`0.0224`, 11위) 등 기존 핵심 위험 요인들이 모델 분기에서도 일관되게 높은 기여도를 보임.

* **하위권 파생 변수 3종의 의미 해석 (`Unhealthy_Lifestyle_Score` 0.0055, `Cardio_DiffWalk_Inter` 0.0048, `Metabolic_BadLife_Inter` 0.0025):**
  * **분기의 중복·우선순위 밀림:** 중요도가 낮다는 것이 당뇨와 무관하다는 뜻은 아니며, 이미 `GenHlth_BMI_Inter`, `Metabolic_Risk_Count`, `Age_HighBP_Inter`, `GenHlth`, `HighBP` 등 상위 핵심 변수들이 주된 위험 구조를 먼저 설명하면서 추가 분기 이득이 작게 집계된 결과임.
  * **변수의 역할 분리 (예측 주력 vs. 고위험군 프로파일링):** 대사·건강·연령 결합 변수는 모델 예측의 핵심 분기를 담당하고, `Cardio_DiffWalk_Inter`나 `Metabolic_BadLife_Inter`처럼 표본 비중이 작지만 유병률(`30~40%`대)이 높은 교차 변수는 초고위험 세부 집단을 설명·군집화하는 인사이트 지표로 활용하는 것이 적절함.

* **모델링 인사이트 및 다음 방향:** 파생 변수는 전체 `ROC-AUC` 수치를 급등시키기보다 모델의 핵심 분기 기준을 직관적인 복합 위험 지표로 재구성하는 효과가 뚜렷하므로, 이후 경량화 실험 시 하위 교차 변수를 정리해 성능을 점검하고 `XGBoost`·`LightGBM`과도 동일한 기준으로 비교해 볼 수 있음.

(참고) 트리 계열 모델이 '비선형 패턴'을 잡는 원리구간 분할(분기) 구조: 
- 트리 모델은 데이터를 "이 변수 값이 기준보다 큰가/작은가?" 같은 질문으로 여러 단계에 걸쳐 잘라 나감.   
- 계단식·조합 경계 형성: 여러 번 조건을 나누다 보면 결과 경계가 직선이 아니라 계단 모양이나 구역별 복합 규칙으로 만들어짐. 그래서 변수 간 조합이나 구간별 차이 같은 비선형 패턴을 별도의 곡선 식 없이도 비교적 자연스럽게 학습할 수 있음.  

In [3]:
pip install xgboost lightgbm

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   -- ------------------------------------- 3.7/48.9 MB 24.9 MB/s eta 0:00:02
   ------ --------------------------------- 8.4/48.9 MB 21.2 MB/s eta 0:00:02
   --------- ------------------------------ 11.5/48.9 MB 18.0 MB/s eta 0:00:03
   ---------------- ----------------------- 20.4/48.9 MB 24.1 MB/s eta 0:00:02
   ------------------------- -------------- 31.5/48.9 MB 29.6 MB/s eta 0:00:01
   ---------------------------------- ----- 41.9/48.9 MB 32.9 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 34.2 MB/s eta 0:00:01
   ---------------------------------------- 48.9/48.9 MB 30.1 MB/s  0:00:01
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 39.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [xgboost]
   ---------------------------------------- 0/2 [xgboost]
   ----------------------------------------


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## xgboost와 lightgbm 설치


In [4]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. 정제된 데이터 로드 (notebooks/ 기준 상위 폴더 경로)
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
X = df.drop(columns=[target_col])
y = df[target_col]

# 2. 전처리 파이프라인 (연속형/수치형 스케일링은 로지스틱 회귀 안정화용)
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[("scaler", StandardScaler(), numeric_cols)],
    remainder="passthrough",
)


def build_models(y_fold_train: pd.Series):
  pos_count = (y_fold_train == 1).sum()
  neg_count = (y_fold_train == 0).sum()
  pos_weight = (neg_count / pos_count) if pos_count > 0 else 1.0

  models = {
      "LogisticRegression (Baseline)": Pipeline([
          ("prep", preprocessor),
          (
              "clf",
              LogisticRegression(
                  max_iter=1000, class_weight="balanced", random_state=42
              ),
          ),
      ]),
      "RandomForest": RandomForestClassifier(
          n_estimators=200,
          max_depth=12,
          min_samples_leaf=20,
          class_weight="balanced",
          n_jobs=-1,
          random_state=42,
      ),
  }

  try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )
  except ImportError:
    print("[안내] xgboost가 설치되지 않아 건너뜁니다.")

  try:
    from lightgbm import LGBMClassifier

    models["LightGBM"] = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
  except ImportError:
    print("[안내] lightgbm이 설치되지 않아 건너뜁니다.")

  return models


# 3. 동일한 5-Fold Stratified K-Fold 분할로 전체 모델 비교
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_names = list(build_models(y).keys())
results = []

for model_name in model_names:
  train_aucs, val_aucs = [], []
  f1s, recalls, precisions = [], [], []

  for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = build_models(y_train)[model_name]
    model.fit(X_train, y_train)

    train_prob = model.predict_proba(X_train)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)

    train_aucs.append(roc_auc_score(y_train, train_prob))
    val_aucs.append(roc_auc_score(y_val, val_prob))
    f1s.append(f1_score(y_val, y_pred, zero_division=0))
    recalls.append(recall_score(y_val, y_pred, zero_division=0))
    precisions.append(precision_score(y_val, y_pred, zero_division=0))

  mean_train = np.mean(train_aucs)
  mean_val = np.mean(val_aucs)
  std_val = np.std(val_aucs)

  results.append({
      "Model": model_name,
      "Train_AUC": f"{mean_train:.4f}",
      "Val_AUC (mean±std)": f"{mean_val:.4f} ± {std_val:.4f}",
      "AUC_Gap": f"{(mean_train - mean_val):.4f}",
      "F1-score": f"{np.mean(f1s):.4f}",
      "Recall": f"{np.mean(recalls):.4f}",
      "Precision": f"{np.mean(precisions):.4f}",
  })

summary_df = pd.DataFrame(results)
print("\n=== [3단계 보완] 전체 모델 5-Fold 비교 ===")
print(summary_df.to_string(index=False))


=== [3단계 보완] 전체 모델 5-Fold 비교 ===
                        Model Train_AUC Val_AUC (mean±std) AUC_Gap F1-score Recall Precision
LogisticRegression (Baseline)    0.8242    0.8241 ± 0.0028  0.0001   0.4433 0.7669    0.3118
                 RandomForest    0.8463    0.8277 ± 0.0031  0.0186   0.4426 0.7833    0.3085
                      XGBoost    0.8483    0.8290 ± 0.0027  0.0194   0.4431 0.7890    0.3081
                     LightGBM    0.8441    0.8297 ± 0.0029  0.0144   0.4423 0.7941    0.3065


## 📌 당뇨병 예측 모델링(3~4단계) 핵심해설 및 인사이트 메모


* **전체 4종 모델(5-Fold Stratified CV) 판별력 비교:**
  * **부스팅 계열 우위:** `LightGBM` (`Val ROC-AUC 0.8297 ± 0.0029`)과 `XGBoost` (`0.8290 ± 0.0027`)가 가장 높은 일반화 판별력을 기록했고, 배깅 계열인 `RandomForest` (`0.8277 ± 0.0031`)도 베이스라인(`LogisticRegression`, `0.8241 ± 0.0028`) 대비 `+0.0036` 상승하며 비선형 분기 구조의 이점을 입증함.
  * **과적합(`AUC_Gap`) 안정성:** `LightGBM`은 `Train_AUC 0.8441`, `Val_AUC 0.8297`로 `AUC_Gap`이 `0.0144`에 그쳐 트리 모델 중 가장 안정적이었으며, `RandomForest` (`0.0186`)와 `XGBoost` (`0.0194`)도 깊이·리프 제약 및 학습률 설정으로 과도한 과적합을 억제함.
  * **분류 지표(`Recall`, `F1-score`, `Precision`) 흐름:** 실제 당뇨 양성자를 포착하는 `Recall`은 `LightGBM` (`0.7941`)과 `XGBoost` (`0.7890`)가 `RandomForest` (`0.7833`) 및 베이스라인 (`0.7669`)보다 높게 나타났고, `F1-score` (`0.4423~0.4433`)와 `Precision` (`0.3065~0.3118`)은 네 모델 간 큰 차이 없이 유사한 범위를 유지함.


* **핵심 인사이트 및 활용 방향:**
  * 선형 베이스라인 대비 비선형 트리 모델(`LightGBM`, `XGBoost`, `RandomForest`)이 복합·구간 패턴을 더 잘 포착하고 `Recall`을 끌어올렸으므로, 양성 놓침(`FN`) 최소화가 중요한 선별 목적에서는 부스팅 계열(`LightGBM`, `XGBoost`)이 유리한 후보가 됨.
  * 변수 역할 분리 관점에서 대사·건강·연령 결합 지표는 예측 모델의 핵심 분기 축으로 쓰고, 유병률이 높은 세부 교차 변수(`Cardio_DiffWalk_Inter`, `Metabolic_BadLife_Inter`)는 고위험군 프로파일링 및 설명 지표로 활용하는 전략이 유효함.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

# 1. 정제된 데이터 로드
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col]
X_raw = df.drop(columns=[target_col])


# 2. 파생 변수 8개 생성
def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  d = data.copy()

  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X_fe = add_interaction_features(X_raw)
derived_cols = [c for c in X_fe.columns if c not in X_raw.columns]

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


def make_boosting_models(y_fold_train: pd.Series):
  pos_count = (y_fold_train == 1).sum()
  neg_count = (y_fold_train == 0).sum()
  pos_weight = (neg_count / pos_count) if pos_count > 0 else 1.0

  return {
      "XGBoost": XGBClassifier(
          n_estimators=300,
          learning_rate=0.05,
          max_depth=6,
          scale_pos_weight=pos_weight,
          eval_metric="logloss",
          random_state=42,
          n_jobs=-1,
      ),
      "LightGBM": LGBMClassifier(
          n_estimators=300,
          learning_rate=0.05,
          class_weight="balanced",
          random_state=42,
          n_jobs=-1,
          verbose=-1,
      ),
  }


# 3. 5-Fold CV 점수 및 Feature Importance 동시 계산
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
score_rows = []
importances_dict = {}

for name in ["XGBoost", "LightGBM"]:
  tr_aucs, va_aucs, f1s, recs, precs = [], [], [], [], []
  fold_imps = []

  for tr_idx, va_idx in skf.split(X_fe, y):
    X_tr, X_va = X_fe.iloc[tr_idx], X_fe.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = make_boosting_models(y_tr)[name]
    model.fit(X_tr, y_tr)

    tr_prob = model.predict_proba(X_tr)[:, 1]
    va_prob = model.predict_proba(X_va)[:, 1]
    va_pred = model.predict(X_va)

    tr_aucs.append(roc_auc_score(y_tr, tr_prob))
    va_aucs.append(roc_auc_score(y_va, va_prob))
    f1s.append(f1_score(y_va, va_pred, zero_division=0))
    recs.append(recall_score(y_va, va_pred, zero_division=0))
    precs.append(precision_score(y_va, va_pred, zero_division=0))
    fold_imps.append(model.feature_importances_)

  mean_tr = np.mean(tr_aucs)
  mean_va = np.mean(va_aucs)
  std_va = np.std(va_aucs)

  score_rows.append({
      "Model (With_Derived)": name,
      "Train_AUC": f"{mean_tr:.4f}",
      "Val_AUC (mean±std)": f"{mean_va:.4f} ± {std_va:.4f}",
      "AUC_Gap": f"{(mean_tr - mean_va):.4f}",
      "F1": f"{np.mean(f1s):.4f}",
      "Recall": f"{np.mean(recs):.4f}",
      "Precision": f"{np.mean(precs):.4f}",
  })

  importances_dict[name] = np.mean(fold_imps, axis=0)

print("=== [파생 변수 포함 5-Fold 점수] XGBoost & LightGBM ===")
print(pd.DataFrame(score_rows).to_string(index=False))

for name, mean_imp in importances_dict.items():
  imp_df = pd.DataFrame({
      "Feature": X_fe.columns,
      "Importance": mean_imp,
      "Type": [
          "Derived" if c in derived_cols else "Original" for c in X_fe.columns
      ],
  }).sort_values("Importance", ascending=False)

  print(f"\n=== [{name}] Feature Importance Top 15 (파생 변수 포함) ===")
  print(imp_df.head(15).to_string(index=False))

=== [파생 변수 포함 5-Fold 점수] XGBoost & LightGBM ===
Model (With_Derived) Train_AUC Val_AUC (mean±std) AUC_Gap     F1 Recall Precision
             XGBoost    0.8483    0.8292 ± 0.0025  0.0191 0.4435 0.7896    0.3084
            LightGBM    0.8450    0.8298 ± 0.0027  0.0152 0.4431 0.7937    0.3073

=== [XGBoost] Feature Importance Top 15 (파생 변수 포함) ===
              Feature  Importance     Type
 Metabolic_Risk_Count    0.470031  Derived
    GenHlth_BMI_Inter    0.094936  Derived
     Age_HighBP_Inter    0.093906  Derived
     Aged_Poor_Health    0.042533  Derived
                  Age    0.035174 Original
             HighChol    0.027873 Original
            CholCheck    0.023864 Original
 HeartDiseaseorAttack    0.023383 Original
    HvyAlcoholConsump    0.022343 Original
             DiffWalk    0.018210 Original
                  Sex    0.016383 Original
               HighBP    0.014630 Original
              GenHlth    0.012004 Original
               Income    0.011662 Original
Cardi

** 이미 RandomForest에서 전체적인 방향성과 하위 파생 변수의 해석을 마쳤으므로, LightGBM과 XGBoost는 파생 변수 포함 기준 5-Fold 점수와 Top 10~15 Feature Importance만 가볍게 뽑아 대조

In [2]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 1000)

# 1. 정제된 데이터 로드 및 파생 변수 8개 생성
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col]
X_raw = df.drop(columns=[target_col])


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  d = data.copy()

  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X_fe = add_interaction_features(X_raw)
derived_cols = [c for c in X_fe.columns if c not in X_raw.columns]

# 2. 5-Fold 평균 LightGBM Feature Importance 계산
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_imps = []

for tr_idx, _ in skf.split(X_fe, y):
  X_tr, y_tr = X_fe.iloc[tr_idx], y.iloc[tr_idx]
  lgbm = LGBMClassifier(
      n_estimators=300,
      learning_rate=0.05,
      class_weight="balanced",
      random_state=42,
      n_jobs=-1,
      verbose=-1,
  )
  lgbm.fit(X_tr, y_tr)
  fold_imps.append(lgbm.feature_importances_)

lgbm_imp_df = (
    pd.DataFrame({
        "Feature": X_fe.columns,
        "Importance (Split Count)": np.mean(fold_imps, axis=0),
        "Type": [
            "Derived" if c in derived_cols else "Original" for c in X_fe.columns
        ],
    })
    .sort_values("Importance (Split Count)", ascending=False)
    .reset_index(drop=True)
)

print("=== [LightGBM] Feature Importance Top 15 (전체) ===")
for rank, row in lgbm_imp_df.head(15).iterrows():
  print(
      f"{rank+1:2d}. {row['Feature']:26s} | {row['Importance (Split Count)']:6.1f} | {row['Type']}"
  )

=== [LightGBM] Feature Importance Top 15 (전체) ===
 1. GenHlth_BMI_Inter          | 1505.2 | Derived
 2. Age                        | 1068.4 | Original
 3. BMI                        |  969.2 | Original
 4. Income                     |  649.8 | Original
 5. PhysHlth                   |  561.6 | Original
 6. MentHlth                   |  550.2 | Original
 7. Education                  |  401.8 | Original
 8. Age_HighBP_Inter           |  370.2 | Derived
 9. Sex                        |  312.2 | Original
10. Unhealthy_Lifestyle_Score  |  311.6 | Derived
11. Metabolic_Risk_Count       |  307.2 | Derived
12. CholCheck                  |  208.6 | Original
13. HvyAlcoholConsump          |  206.0 | Original
14. HighChol                   |  201.2 | Original
15. GenHlth                    |  180.4 | Original


## 📌 파생 변수 8개 추가 시 XGBoost · LightGBM 5-Fold 점수 및 상위 변수 중요도 요약


* **파생 변수 8개 추가 후 5-Fold 성능 요약:**
  * `LightGBM`: `Train_AUC 0.8450`, `Val_AUC 0.8298 ± 0.0027`, `AUC_Gap 0.0152`, `F1 0.4431`, `Recall 0.7937`, `Precision 0.3073`으로 원본 대비 최고 수준의 일반화 판별력과 낮은 폴드 편차를 유지함.
  * `XGBoost`: `Train_AUC 0.8483`, `Val_AUC 0.8292 ± 0.0025`, `AUC_Gap 0.0191`, `F1 0.4435`, `Recall 0.7896`, `Precision 0.3084`로 원본 대비 `Val_AUC`와 `F1`, `Recall`이 소폭 상승하며 안정적인 일반화를 보임.
  * 파생 변수 8개는 점수를 급격히 끌어올리기보다 비선형 분할 모델의 분기 구조를 정교화하고 기존 판별력·재현율을 뒷받침하는 역할을 수행함.

* **XGBoost 상위 Feature Importance 해석 (비율 중요도 기준):**
  * `Metabolic_Risk_Count` (`0.4700`, 1위), `GenHlth_BMI_Inter` (`0.0949`, 2위), `Age_HighBP_Inter` (`0.0939`, 3위), `Aged_Poor_Health` (`0.0425`, 4위)가 1~4위를 차지하며 대사 위험 누적과 건강·연령 결합 변수가 핵심 분기를 주도함.
  * `Age`, `HighChol`, `DiffWalk`, `HighBP`, `GenHlth` 등 주요 위험 요인과 `Cardio_DiffWalk_Inter` (`0.0109`)도 상위 15위 내에 함께 포함됨.

* **LightGBM 상위 Feature Importance 해석 (분할 횟수 Split 기준):**
  * `GenHlth_BMI_Inter` (`1505.2`, 1위)가 압도적 1위를 기록했고, `Age_HighBP_Inter` (`370.2`, 8위), `Unhealthy_Lifestyle_Score` (`311.6`, 10위), `Metabolic_Risk_Count` (`307.2`, 11위) 등 파생 변수들이 분기에 활용됨.
  * `Age` (`1068.4`, 2위), `BMI` (`969.2`, 3위), `Income` (`649.8`, 4위), `PhysHlth` (`561.6`, 5위), `MentHlth` (`550.2`, 6위), `Education` (`401.8`, 7위) 등 연속·순서형 변수들의 분할 횟수가 높게 나타나 구간을 세밀하게 쪼개는 부스팅 특성이 반영됨.

* **3개 트리 모델(RandomForest · XGBoost · LightGBM) 교차 대조 인사이트:**
  * 모델별 중요도 산출 방식(`RandomForest`·`XGBoost` 비율 중요도 vs. `LightGBM` 분할 횟수) 차이는 있으나, `GenHlth_BMI_Inter`, `Metabolic_Risk_Count`, `Age_HighBP_Inter` 및 `Age`, `BMI` 등 핵심 위험 축이 반복적으로 상위권에 등장해 변수 중요도 해석의 일관성을 뒷받침함.
  * `LightGBM` 하위 Top 15에 보이는 `CholCheck` (`208.6`), `HvyAlcoholConsump` (`206.0`) 등은 행동·검진 특성이 반영된 항목이므로 당뇨 직접 원인으로 단정하지 않고 보수적으로 해석하는 것이 바람직함.

** (참고 요약)
높은 분할 횟수 ≠ 직접적 원인: LightGBM 중요도는 "예측에 얼마나 자주 활용되었는가"를 보여줄 뿐, 인과관계를 증명하지는 않음.

해석 원칙: CholCheck와 HvyAlcoholConsump는 검진·진단 여부 및 사후 행동 변화·편향 가능성을 함께 명시해 조심스럽게 해석하고, 핵심 위험 요인(GenHlth, HighBP, BMI, Age 및 대사·건강 결합 변수) 중심으로 설득력 있게 정리하는 것이 타당함.

## 📌 3개 트리 모델(RandomForest · XGBoost · LightGBM) 변수 중요도 한눈에 비교

| 순위 기준 | `RandomForest` (비율 중요도) | `XGBoost` (비율 중요도) | `LightGBM` (분할 횟수 Split) |
| :--- | :--- | :--- | :--- |
| **1위** | `GenHlth_BMI_Inter` (`0.2004`) | `Metabolic_Risk_Count` (`0.4700`) | `GenHlth_BMI_Inter` (`1505.2`) |
| **2위** | `Metabolic_Risk_Count` (`0.1543`) | `GenHlth_BMI_Inter` (`0.0949`) | `Age` (`1068.4`) |
| **3위** | `Age_HighBP_Inter` (`0.1097`) | `Age_HighBP_Inter` (`0.0939`) | `BMI` (`969.2`) |

| **Top 15 파생 변수** | `GenHlth_BMI_Inter`, `Metabolic_Risk_Count`, `Age_HighBP_Inter`, `Aged_Poor_Health`, `Triple_Metabolic_Risk` | `Metabolic_Risk_Count`, `GenHlth_BMI_Inter`, `Age_HighBP_Inter`, `Aged_Poor_Health`, `Cardio_DiffWalk_Inter` | `GenHlth_BMI_Inter`, `Age_HighBP_Inter`, `Unhealthy_Lifestyle_Score`, `Metabolic_Risk_Count` |
| **주요 공통 핵심 축** | `GenHlth_BMI_Inter`, `Metabolic_Risk_Count`, `Age_HighBP_Inter`, `Age`, `BMI`, `HighBP`, `GenHlth` | `Metabolic_Risk_Count`, `GenHlth_BMI_Inter`, `Age_HighBP_Inter`, `Aged_Poor_Health`, `Age`, `HighChol` | `GenHlth_BMI_Inter`, `Age_HighBP_Inter`, `Metabolic_Risk_Count`, `Age`, `BMI` |

* **공통 핵심 축 일치:** 세 모델 모두 `GenHlth_BMI_Inter`, `Metabolic_Risk_Count`, `Age_HighBP_Inter` 같은 대사·건강·연령 결합 파생 변수와 `Age`, `BMI` 등 기존 주요 위험 요인을 핵심 분기 기준으로 삼아 4단계 변수 중요도 교차 검증의 일관성을 뒷받침함.

* **모델별 중요도 산출·분할 특성 차이:** `RandomForest`와 `XGBoost`는 비율 중요도(합 `1.0`) 기준이며 `XGBoost`는 `Metabolic_Risk_Count`(`0.4700`)에 큰 비중을 둔 반면, `LightGBM`은 트리 분할 횟수(Split Count) 기준이라 연속·순서형 변수(`Age`, `BMI`, `Income`, `PhysHlth`, `MentHlth`, `Education`)의 세분화된 분기 횟수가 상위에 함께 나타남.

* **하위·주의 변수 해석 원칙:** `RandomForest` 하위권 파생 변수(`Unhealthy_Lifestyle_Score` `0.0055`, `Cardio_DiffWalk_Inter` `0.0048`, `Metabolic_BadLife_Inter` `0.0025`)는 상위 변수가 핵심 위험 구조를 먼저 설명하거나 표본 비중이 작은 교차 집단 특성이 반영된 결과이며, `LightGBM` 12~15위권의 `CholCheck` (`208.6`)·`HvyAlcoholConsump` (`206.0`) 등은 검진·진단 여부 및 사후 행동 변화 가능성을 함께 고려해 해석해야 함.


## 📌 당뇨병 예측 모델링(3~4단계) 최종 종합 요약 메모


* **전체 4종 모델(5-Fold Stratified CV) 성능 비교:**
  * `LightGBM` (`Val ROC-AUC 0.8297 ± 0.0029`, `AUC_Gap 0.0144`, `Recall 0.7941`, `Precision 0.3065`, `F1 0.4423`)과 `XGBoost` (`Val ROC-AUC 0.8290 ± 0.0027`, `AUC_Gap 0.0194`, `Recall 0.7890`, `Precision 0.3081`, `F1 0.4431`)가 베이스라인 `LogisticRegression` (`Val ROC-AUC 0.8241 ± 0.0028`, `AUC_Gap 0.0001`, `Recall 0.7669`, `Precision 0.3118`, `F1 0.4433`) 및 배깅 `RandomForest` (`Val ROC-AUC 0.8277 ± 0.0031`, `AUC_Gap 0.0186`, `Recall 0.7833`, `Precision 0.3085`, `F1 0.4426`) 대비 일반화 판별력과 양성 탐지(`Recall`)에서 우위를 보임.
  * 트리 계열 모델들은 비선형 구간·상호작용 경계를 포착하면서 `AUC_Gap`이 `0.0144~0.0194` 수준으로 제어되어 과도한 과적합 없이 안정적인 일반화 성능을 나타냄.

* **파생 변수 8개 추가 효과 (3단계 모델별 대조):**
  * `RandomForest`: `Val ROC-AUC` `0.8277 ± 0.0031` → `0.8277 ± 0.0029`로 동일 유지, `F1` (`0.4426` → `0.4437`)·`Precision` (`0.3085` → `0.3103`) 소폭 상승, `Recall` (`0.7833` → `0.7783`) 미세 감소.
  * `LightGBM`: 파생 변수 포함 시 `Train_AUC 0.8450`, `Val_AUC 0.8298 ± 0.0027`, `AUC_Gap 0.0152`, `F1 0.4431`, `Recall 0.7937`, `Precision 0.3073`으로 최고 판별력과 재현율을 안정적으로 유지.
  * `XGBoost`: 파생 변수 포함 시 `Train_AUC 0.8483`, `Val_AUC 0.8292 ± 0.0025`, `AUC_Gap 0.0191`, `F1 0.4435`, `Recall 0.7896`, `Precision 0.3084`로 `Val ROC-AUC`·`F1`·`Recall`이 미세 상승함.
  * 파생 변수는 전체 `ROC-AUC` 수치를 급등시키기보다 모델의 분기 구조를 정교화하고 해석 가능한 복합 위험 지표로 재구성하는 데 주된 역할이 있음.

* **4단계 `Feature Importance` 교차 검증 및 변수 해석:**
  * **세 트리 모델 공통 핵심 축:** `GenHlth_BMI_Inter`, `Metabolic_Risk_Count`, `Age_HighBP_Inter` 같은 대사·건강·연령 결합 파생 변수와 `Age`, `BMI` 등 기존 핵심 위험 요인이 분기 기준에서 일관되게 중시됨 (`RandomForest` 1위 `GenHlth_BMI_Inter` `0.2004`, `XGBoost` 1위 `Metabolic_Risk_Count` `0.4700`, `LightGBM` Split 1위 `GenHlth_BMI_Inter` `1505.2`).
  * **하위 파생 변수 의미 (`Unhealthy_Lifestyle_Score` `0.0055`, `Cardio_DiffWalk_Inter` `0.0048`, `Metabolic_BadLife_Inter` `0.0025`):** 당뇨와 무관하다는 뜻이 아니라 상위 핵심 변수가 주된 위험 구조를 먼저 설명하면서 추가 분기 이득이 작게 집계되었거나 표본 비중이 작은 세부 교차 집단 특성이 반영된 결과임.
  * **주의 변수 해석 원칙:** `LightGBM` 12~15위권의 `CholCheck` (`208.6`)·`HvyAlcoholConsump` (`206.0`) 등은 검진·진단 여부 및 사후 행동 변화·편향 가능성을 함께 고려해야 하며, 분할 횟수(Split Count) 순위 자체를 직접적 인과관계로 단정하지 않도록 유의함.

* **핵심 인사이트 및 후속 방향:**
  * 당뇨 고위험군 선별(`Recall` 확보 및 `ROC-AUC` 판별력) 측면에서는 부스팅 계열(`LightGBM`, `XGBoost`)이 유력한 후보가 되며, 대사·건강·연령 결합 변수는 예측 주력 분기로, 고유병률 세부 교차 변수는 고위험군 프로파일링 지표로 역할을 분리해 활용할 수 있음.
  * 기본 임계점(`0.5`) 기준 비교 결과는 기준선으로 보존한 뒤, 후속 검증 단계에서 운영 목적(선별 `Recall` 중심 vs. `F1` 균형)에 맞춘 임계점(Threshold) 조정과 피처 경량화 실험을 이어갈 수 있음.